In [ ]:
import os
import re
import json
import time
import hashlib
import requests
import feedparser
import math
import datetime as dt
import datetime as dt
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

# =========================
# Caminhos do run atual
# =========================

PROJECT_ROOT = Path(".").resolve()
RUN_DATE = dt.date.today().isoformat()
RUN_DIR = PROJECT_ROOT / "runs" / RUN_DATE

RAW_DIR = RUN_DIR / "raw"
BATCH_DIR = RUN_DIR / "batches"
RESP_DIR = RUN_DIR / "responses"
FINAL_DIR = RUN_DIR / "final"

CONFIG_PATH = PROJECT_ROOT / "sources_config.json"  # JSON guia
STATE_DIR = PROJECT_ROOT / "state"
STATE_DIR.mkdir(parents=True, exist_ok=True)

STATE_PATH = STATE_DIR / "seen.json"


RUN_DIR = PROJECT_ROOT / "runs" / RUN_DATE
RAW_DIR = RUN_DIR / "raw"
BATCH_DIR = RUN_DIR / "batches"
RESP_DIR = RUN_DIR / "responses"
FINAL_DIR = RUN_DIR / "final"

def today_str_local() -> str:
    return dt.date.today().isoformat()

RUN_DATE = today_str_local()

FIRECRAWL_API_KEY = os.getenv("FIRECRAWL_API_KEY", "")
FIRECRAWL_BASE_URL = os.getenv("FIRECRAWL_BASE_URL", "http://firecrawl")

DIFY_BASE_URL = "http://DIFY"
DIFY_API_KEY = "apikey"
DIFY_USER = "product-radar"



In [86]:
def utc_now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat() + "Z"

for d in [RAW_DIR, BATCH_DIR, RESP_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def load_config() -> Dict[str, Any]:
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def load_state() -> Dict[str, Any]:
    if not STATE_PATH.exists():
        return {"seen_urls": {}, "seen_hashes": {}}
    with open(STATE_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_state(state: Dict[str, Any]) -> None:
    with open(STATE_PATH, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

config = load_config()
state = load_state()

print("RUN_DIR:", RUN_DIR)
print("Loaded sources:", len(config.get("sources", [])))

RUN_DIR: /mnt/data/jupyter-env/dbaas_Intelligence_Radar/runs/2026-03-13
Loaded sources: 17


In [87]:
def sha256_text(text: str) -> str:
    h = hashlib.sha256()
    h.update(text.encode("utf-8", errors="ignore"))
    return h.hexdigest()

def normalize_text(text: str) -> str:
    """
    Normalização mais estável para delta por conteúdo.
    Remove ruídos típicos de páginas web e reduz falsos positivos.
    """
    if not text:
        return ""

    t = text

    # normaliza quebras de linha
    t = t.replace("\r\n", "\n").replace("\r", "\n")

    # remove linhas típicas de "updated" e timestamps
    t = re.sub(r"(?im)^\s*(last\s+updated|updated\s+on|updated|last\s+modified)\s*:?.*$", "", t)

    # remove datas ISO soltas 
    t = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", "", t)

    # colapsa espaços
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)

    return t.strip()

def seen_url(url: str, state: Dict[str, Any]) -> bool:
    """
    True se a URL já foi vista antes.
    """
    return bool(url and url in state.get("seen_urls", {}))

def already_seen(url: str, content_hash: Optional[str], state: Dict[str, Any]) -> bool:
    """
    Delta por conteúdo:
    - sem URL e com hash: usa seen_hashes
    - com URL e hash:
        - URL nunca vista -> novo
        - URL vista + mesmo hash -> já visto
        - URL vista + hash diferente -> update
    - com URL e sem hash:
        - não decide por conteúdo
        - para gate estrito por URL, use seen_url(url, state)
    """
    if not url:
        return bool(content_hash and content_hash in state.get("seen_hashes", {}))

    url_entry = state.get("seen_urls", {}).get(url)

    if not url_entry:
        return False

    if not content_hash:
        return False

    prev_hash = None
    if isinstance(url_entry, dict):
        prev_hash = url_entry.get("last_hash")
    elif isinstance(url_entry, str):
        prev_hash = None

    if prev_hash and prev_hash == content_hash:
        return True

    return False

def mark_seen(url: str, content_hash: str, state: Dict[str, Any]) -> None:
    """
    Novo state:
    - seen_urls[url] vira um dict com last_hash + timestamps
    - seen_hashes continua existindo só para dedup cross-source (opcional)
    """
    ts = utc_now_iso()

    if url:
        existing = state.get("seen_urls", {}).get(url)

        if isinstance(existing, str) or existing is None:
            state["seen_urls"][url] = {
                "first_seen_at": existing or ts,
                "last_seen_at": ts,
                "last_hash": content_hash,
                "update_count": 0,
            }
        else:
            # já é dict
            existing["last_seen_at"] = ts
            prev_hash = existing.get("last_hash")
            if prev_hash and prev_hash != content_hash:
                existing["update_count"] = int(existing.get("update_count", 0)) + 1
            existing["last_hash"] = content_hash

    # mantém dedup global por hash
    if content_hash:
        state.setdefault("seen_hashes", {})
        if content_hash not in state["seen_hashes"]:
            state["seen_hashes"][content_hash] = ts

In [89]:
def parse_rss(url: str, max_items: int = 50) -> List[Dict[str, Any]]:
    feed = feedparser.parse(url)
    items = []
    for entry in (feed.entries or [])[:max_items]:
        link = getattr(entry, "link", None) or ""
        title = getattr(entry, "title", None) or ""
        published = ""
        if getattr(entry, "published_parsed", None):
            published = dt.datetime(*entry.published_parsed[:6]).replace(microsecond=0).isoformat() + "Z"
        elif getattr(entry, "updated_parsed", None):
            published = dt.datetime(*entry.updated_parsed[:6]).replace(microsecond=0).isoformat() + "Z"

        items.append({
            "url": link,
            "title": title,
            "published_at": published
        })
    return items

In [90]:
def firecrawl_scrape(url: str) -> Dict[str, Any]:
    """
    Esperado retornar markdown limpo e metadados.
    Ajuste o endpoint e formato conforme seu Firecrawl.
    """
    endpoint = f"{FIRECRAWL_BASE_URL}/v1/scrape"
    headers = {
        "Authorization": f"Bearer {FIRECRAWL_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "url": url,
        "formats": ["markdown"],
        "onlyMainContent": True
    }
    r = requests.post(endpoint, headers=headers, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()

In [91]:
def firecrawl_extract_links(index_url: str, allow_prefixes: Optional[List[str]] = None, limit: int = 30) -> List[str]:
    """
    Extrai links da página índice usando Firecrawl scrape em markdown e regex simples.
    Isso é suficiente para blog index em muitos casos.
    """
    data = firecrawl_scrape(index_url)
    md = ""
    if isinstance(data, dict):
        md = (
            data.get("data", {}).get("markdown")
            or data.get("markdown")
            or ""
        )
    md = md or ""
    candidates = re.findall(r"\((https?://[^)]+)\)", md)
    urls = []
    for u in candidates:
        u = u.strip()
        if allow_prefixes and not any(u.startswith(p) for p in allow_prefixes):
            continue
        if u not in urls:
            urls.append(u)
        if len(urls) >= limit:
            break
    return urls

In [92]:
def build_item(
    source: Dict[str, Any],
    url: str,
    title: str,
    published_at: str,
    content_markdown: str
) -> Dict[str, Any]:
    content_markdown = normalize_text(content_markdown)

    # mudou aqui: se não tem conteúdo, não inventa hash por URL
    content_hash = sha256_text(content_markdown) if content_markdown else None

    # mudou aqui: item_id continua estável, mas agora não depende do hash falso
    basis = url + "|" + (content_hash or "no-content")
    item_id = f"sha256:{sha256_text(basis)}"

    return {
        "id": item_id,
        "source_id": source["id"],
        "source_name": source["name"],
        "category": source["category"],
        "tier": source["tier"],
        "url": url,
        "title": title,
        "published_at": published_at,
        "collected_at": utc_now_iso(),
        "content_markdown": content_markdown,
        "content_hash": content_hash,
        "tags": config.get("tags", {}).get(source["category"], []),
    }

In [113]:
def collect_source_items(source: Dict[str, Any], state: Dict[str, Any]) -> List[Dict[str, Any]]:
    method = source.get("ingest", {}).get("method")
    max_items = config.get("run_policy", {}).get("max_items_per_source_per_run", 20)

    collected: List[Dict[str, Any]] = []

    if method == "rss":
        feed_items = parse_rss(source["url"], max_items=max_items)
        for fi in feed_items:
            url = fi["url"]
            if not url:
                continue

            # Para RSS, delta por URL
            if seen_url(url, state):
                continue

            scraped = firecrawl_scrape(url)
            md = (
                scraped.get("data", {}).get("markdown")
                or scraped.get("markdown")
                or ""
            )
            md = md or ""
            if len(md) < config.get("extract_policy", {}).get("content_min_chars", 400):
                continue

            item = build_item(
                source=source,
                url=url,
                title=fi.get("title", "") or "",
                published_at=fi.get("published_at", "") or "",
                content_markdown=md
            )

            if already_seen(item["url"], item["content_hash"], state):
                continue

            collected.append(item)
            mark_seen(item["url"], item["content_hash"], state)

        return collected

    if method == "html_index":
        allow_prefixes = source.get("ingest", {}).get("discovery", {}).get("allow_url_prefixes")
        urls = firecrawl_extract_links(source["url"], allow_prefixes=allow_prefixes, limit=max_items)
        for url in urls:
            scraped = firecrawl_scrape(url)
            md = (
                scraped.get("data", {}).get("markdown")
                or scraped.get("markdown")
                or ""
            )
            title = (
                scraped.get("data", {}).get("metadata", {}).get("title")
                or scraped.get("metadata", {}).get("title")
                or ""
            )
            published = (
                scraped.get("data", {}).get("metadata", {}).get("publishedTime")
                or scraped.get("metadata", {}).get("publishedTime")
                or ""
            )

            md = md or ""
            if len(md) < config.get("extract_policy", {}).get("content_min_chars", 400):
                continue

            item = build_item(
                source=source,
                url=url,
                title=title,
                published_at=published,
                content_markdown=md
            )

            if already_seen(item["url"], item["content_hash"], state):
                continue

            collected.append(item)
            mark_seen(item["url"], item["content_hash"], state)

        return collected

    if method == "release_index":
        # Mantém como página atual, mas sem bloquear por URL antes do scrape.
        url = source["url"]

        scraped = firecrawl_scrape(url)
        md = (
            scraped.get("data", {}).get("markdown")
            or scraped.get("markdown")
            or ""
        )
        title = source["name"]
        published = ""

        md = md or ""
        if len(md) < config.get("extract_policy", {}).get("content_min_chars", 400):
            return []

        item = build_item(
            source=source,
            url=url,
            title=title,
            published_at=published,
            content_markdown=md
        )

        if already_seen(item["url"], item["content_hash"], state):
            return []

        mark_seen(item["url"], item["content_hash"], state)
        return [item]

    if method == "docs_root":
        # Mantém como página raiz, mas sem bloquear por URL antes do scrape.
        url = source["url"]

        scraped = firecrawl_scrape(url)
        md = (
            scraped.get("data", {}).get("markdown")
            or scraped.get("markdown")
            or ""
        )

        md = md or ""
        if len(md) < config.get("extract_policy", {}).get("content_min_chars", 400):
            return []

        item = build_item(
            source=source,
            url=url,
            title=source["name"],
            published_at="",
            content_markdown=md
        )

        if already_seen(item["url"], item["content_hash"], state):
            return []

        mark_seen(item["url"], item["content_hash"], state)
        return [item]

    return []

In [114]:
all_items: List[Dict[str, Any]] = []

sources = config.get("sources", [])
for s in sources:
    try:
        items = collect_source_items(s, state)
        print(f"{s['id']}: collected {len(items)}")
        all_items.extend(items)
        time.sleep(0.2)
    except Exception as e:
        print(f"Erro em {s['id']}: {e}")

raw_path = RAW_DIR / "items_raw.json"
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(all_items, f, ensure_ascii=False, indent=2)

save_state(state)

print("Total items:", len(all_items))
print("Saved:", raw_path)

pg_planet_home: collected 1
pg_planet_rss: collected 0
pg_release_notes: collected 0
pg_docs: collected 0
cloudnative_pg_blog_rss: collected 0
pganalyze_blog: collected 3
cybertec_pg_rss: collected 0
crunchydata_rss: collected 0
edb_rss: collected 0
citus_rss: collected 0
mysql_home: collected 8
mysql_release_84: collected 0
mysql_planet_rss: collected 3
aws_database_blog_pt: collected 0
aws_aurora_updates_rss: collected 0
ms_techcommunity_pg_rss: collected 0
ms_releasecommunications_azure_rss: collected 0
Total items: 15
Saved: /mnt/data/jupyter-env/dbaas_Intelligence_Radar/runs/2026-03-13/raw/items_raw.json
